# N02 – Ranking y Poda DIARIA

Este notebook:

- Lee un único dataset diario: **`df_st_daily.xlsx`**
- Construye targets forward a **1m / 3m / 6m / 12m** (21/63/126/252 días)
- Rankea variables por **correlación forward**
- Realiza **poda entre features** por correlación (se queda con `kept`)
- Calcula **Information Value** (una hoja por horizonte + agregado)
- Entrena un **Random Forest simple** sobre `kept` y obtiene importancias (SHAP si está disponible; si no, permutation importance)
- Construye **dos datasets finales**:
  - **corto plazo** (1m–3m)
  - **largo plazo** (6m–12m)
- Calcula **Jaccard** entre las selecciones corto vs largo
- Exporta resultados a `results_n02/`

## 0) Setup

Importamos librerías, definimos parámetros y preparamos el directorio de resultados.

In [1]:
import numpy as np
import pandas as pd
import os
from itertools import combinations
from pathlib import Path

RESULTS_DIR = "results_n02"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Horizontes en días hábiles aproximados
HORIZON_MAP = {"1m": 21, "3m": 63, "6m": 126, "12m": 252}

# Selección inicial por horizonte (pool)
TOP_K_PER_H = 250

# Umbral para considerar dos features redundantes (correlación absoluta)
FEATURE_CORR_THRESH = 0.9

# Tamaño final por bloque (corto y largo)
TOP_N_BLOCK = 50

# Pesos del score por bloque (ajustables)
W_CORR = 0.50
W_IV   = 0.20
W_IMP  = 0.30

print("✅ Setup OK")

# Tamaño final general (dataset único)
TOP_N_GENERAL = 50

# Exportar también CSV además de XLSX
EXPORT_CSV = True


✅ Setup OK


## 1) Carga de datos

Leemos `df_st_daily.xlsx`, ordenamos por fecha y mostramos las primeras 5 filas.

In [2]:
BASE_DIR = Path.cwd()

DF_DAILY_ST = BASE_DIR / "results_n01" / "df_st_daily.xlsx"
df = pd.read_excel(DF_DAILY_ST)

df.head(5)

# Si existe una columna llamada 'Fecha', la priorizamos.
if 'Fecha' in df.columns:
    DATE_COL = 'Fecha'
else:
    DATE_COL = df.columns[0]

df = df.sort_values(DATE_COL).reset_index(drop=True)


## 2) Identificar columna del USD y definir features candidatas

- Detectamos una columna que contenga `usd` en el nombre (si hay varias, toma la primera).
- Armamos la lista de columnas candidatas a features (excluyendo fecha y columnas que se generarán luego).

In [3]:
# Definimos explícitamente el target base (Uruguay): usd_dlog
USD_COL = "usd_dlog"
assert USD_COL in df.columns, f"No encontré '{USD_COL}' en el dataset. Columnas disponibles (primeras 20): {df.columns.tolist()[:20]}"

print("USD_COL (target base) =", USD_COL)

# Features candidatas (por ahora: todo menos fecha y USD)
base_feature_cols = [c for c in df.columns if c not in [DATE_COL, USD_COL]]
pd.DataFrame({"feature_cols": base_feature_cols}).head(5)


USD_COL (target base) = usd_dlog


,feature_cols
0,compras_ANCAP
1,int_ANCAP
2,var_stock_pet
3,inv_ANCAP
4,venta_autos


## 3) Construcción de targets forward (retornos y dirección)

Para cada horizonte `h`:
- `ret_fwd_h`: variación del USD log (o nivel) entre `t` y `t+H` (por defecto: `shift(-H) - hoy`)
- `up_fwd_h`: indicador binario (1 si sube, 0 si no sube)

Luego mostramos las primeras 5 filas del dataframe con los nuevos targets.

In [4]:
def forward_sum(arr: np.ndarray, H: int) -> np.ndarray:
    """Suma forward: en t devuelve sum_{i=1..H} arr[t+i]."""
    n = len(arr)
    out = np.full(n, np.nan, dtype=float)
    for t in range(0, n - H):
        out[t] = np.nansum(arr[t+1:t+H+1])
    return out

# Targets forward basados en usd_dlog:
# ret_fwd_h = suma de usd_dlog en los próximos H días (aprox. retorno log acumulado)
usd_arr = df[USD_COL].to_numpy(dtype=float)

for h, H in HORIZON_MAP.items():
    df[f"ret_fwd_{h}"] = forward_sum(usd_arr, H)
    df[f"up_fwd_{h}"]  = (df[f"ret_fwd_{h}"] > 0).astype(int)

df[[DATE_COL, USD_COL] + [f"ret_fwd_{h}" for h in HORIZON_MAP] + [f"up_fwd_{h}" for h in HORIZON_MAP]].head(5)


,Fecha,usd_dlog,ret_fwd_1m,ret_fwd_3m,ret_fwd_6m,ret_fwd_12m,up_fwd_1m,up_fwd_3m,up_fwd_6m,up_fwd_12m
0,2015-04-01,-0.000778,0.025357,0.058195,0.126307,0.185236,1,1,1,1
1,2015-04-06,-0.002727,0.027325,0.068238,0.129377,0.191513,1,1,1,1
2,2015-04-07,0.005059,0.024920,0.062814,0.121915,0.199576,1,1,1,1
3,2015-04-08,0.008889,0.011477,0.053196,0.112339,0.206147,1,1,1,1
4,2015-04-09,0.006902,0.011776,0.044103,0.107841,0.191703,1,1,1,1


## 4) Definir `feature_cols` finales (excluyendo targets)

Excluimos:
- fecha
- USD
- columnas `ret_fwd_*` y `up_fwd_*`

Mostramos las primeras 5 variables.

In [5]:
target_cols = [f"ret_fwd_{h}" for h in HORIZON_MAP] + [f"up_fwd_{h}" for h in HORIZON_MAP]

exclude_cols = {DATE_COL, USD_COL, "dolar", *target_cols}

feature_cols = [c for c in df.columns if c not in exclude_cols]

print("Cantidad de features:", len(feature_cols))
pd.DataFrame({"feature_cols": feature_cols}).head(5)

Cantidad de features: 229


,feature_cols
0,compras_ANCAP
1,int_ANCAP
2,var_stock_pet
3,inv_ANCAP
4,venta_autos


## 5) Ranking por correlación forward (feature vs USD futuro)

Para cada horizonte:
- Calculamos la correlación (Pearson) entre cada feature en `t` y `ret_fwd_h`
- Tomamos el valor absoluto y ordenamos descendentemente
- Guardamos una tabla por horizonte (`rank_tables[h]`)
- Armamos una tabla larga `agg` para cálculos posteriores

Mostramos las primeras 5 filas del ranking del horizonte 1m y las primeras 5 de `agg`.

In [6]:
rank_tables = {}
agg_rows = []

for h in HORIZON_MAP:
    corrs = df[feature_cols].corrwith(df[f"ret_fwd_{h}"]).abs().sort_values(ascending=False)
    t = corrs.reset_index()
    t.columns = ["var", "abs_corr"]
    t["h"] = h
    rank_tables[h] = t
    agg_rows.append(t)

agg = pd.concat(agg_rows, ignore_index=True)

print("Ranking 3m (top 5):")
display(rank_tables["3m"].head(5))

print("agg (head 5):")
agg.head(5)

Ranking 3m (top 5):


,var,abs_corr,h
0,operativa_total_mercado_de_cambios,0.365375,3m
1,tasa_dolares_familias,0.340460,3m
2,dolar_mexico,0.312405,3m
3,uy_bcu_lrm_pct_adjudicado_180d,0.299580,3m
4,EPU_br,0.273734,3m


agg (head 5):


,var,abs_corr,h
0,EPU_br,0.230455,1m
1,tasa_dolares_familias,0.208415,1m
2,exp_madera,0.198079,1m
3,dolar_mexico,0.189519,1m
4,operativa_total_mercado_de_cambios,0.174439,1m


## 6) Export de rankings por horizonte

Guardamos un Excel con:
- una hoja por horizonte `corr_1m`, `corr_3m`, ...
- una hoja `corr_long` con la tabla larga

Mostramos las primeras 5 filas de `agg`.

In [7]:
out_corr = os.path.join(RESULTS_DIR, "correlation_by_horizon.xlsx")
with pd.ExcelWriter(out_corr) as writer:
    for h, t in rank_tables.items():
        t.to_excel(writer, sheet_name=f"corr_{h}", index=False)
    agg.to_excel(writer, sheet_name="corr_long", index=False)

print("✅ Guardado:", out_corr)
agg.head(5)

✅ Guardado: results_n02\correlation_by_horizon.xlsx


,var,abs_corr,h
0,EPU_br,0.230455,1m
1,tasa_dolares_familias,0.208415,1m
2,exp_madera,0.198079,1m
3,dolar_mexico,0.189519,1m
4,operativa_total_mercado_de_cambios,0.174439,1m


## 7) Jaccard diagnóstico (Top-K por correlación)

Calculamos el índice de Jaccard entre los sets `Top-K` de cada horizonte para ver solapamiento.

Mostramos la matriz (primeras 5 filas).

In [8]:
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if len(a | b) else np.nan

TOP_K_JACCARD = 100
h_list = list(HORIZON_MAP.keys())
jac = pd.DataFrame(index=h_list, columns=h_list, dtype=float)

top_sets = {h: rank_tables[h].head(TOP_K_JACCARD)["var"].tolist() for h in h_list}

for h1 in h_list:
    for h2 in h_list:
        jac.loc[h1, h2] = jaccard(top_sets[h1], top_sets[h2])

jac_out = os.path.join(RESULTS_DIR, "jaccard_corr_topk.xlsx")
jac.to_excel(jac_out)
print("✅ Guardado:", jac_out)

jac.head(5)

✅ Guardado: results_n02\jaccard_corr_topk.xlsx


,1m,3m,6m,12m
1m,1.000000,0.515152,0.515152,0.612903
3m,0.515152,1.000000,0.680672,0.639344
6m,0.515152,0.680672,1.000000,0.639344
12m,0.612903,0.639344,0.639344,1.000000


## 8) Pool + poda entre features (redundancia por correlación entre variables)

1) Pool = unión de Top-K por horizonte  
2) Score base para decidir entre dos variables correlacionadas:
   - usamos la **correlación promedio** (abs) sobre horizontes (`abs_corr_mean`)
3) Hacemos selección greedy: mantenemos una variable si su correlación con las ya mantenidas es < umbral.

Resultado:
- `kept`: lista final no redundante
- `dropped_log`: log de eliminaciones

Mostramos:
- primeras 5 variables kept
- primeras 5 filas del log de eliminaciones

In [9]:
# Pool candidato: unión de Top-K por horizonte
pool = set()
for h in HORIZON_MAP:
    pool |= set(rank_tables[h].head(TOP_K_PER_H)["var"].tolist())
pool = list(pool)
print("Pool candidato:", len(pool))

# Score agregado (promedio abs_corr en todos los horizontes)
abs_corr_mean = agg.groupby("var")["abs_corr"].mean()
score_map = abs_corr_mean.to_dict()

# Correlación entre features en el pool
X_pool = df[pool].copy()
corr_pool = X_pool.corr().abs()

pool_sorted = sorted(pool, key=lambda v: score_map.get(v, 0.0), reverse=True)

kept = []
dropped_log = []

for v in pool_sorted:
    if not kept:
        kept.append(v)
        continue

    mx = corr_pool.loc[v, kept].max()
    if pd.isna(mx) or mx < FEATURE_CORR_THRESH:
        kept.append(v)
    else:
        most_sim = corr_pool.loc[v, kept].idxmax()
        dropped_log.append({
            "dropped": v,
            "kept": most_sim,
            "corr_abs": float(mx),
            "dropped_score": float(score_map.get(v, 0.0)),
            "kept_score": float(score_map.get(most_sim, 0.0))
        })

print("Kept tras poda:", len(kept))

out_poda_csv = os.path.join(RESULTS_DIR, "poda_features_corr_log.csv")
pd.DataFrame(dropped_log).to_csv(out_poda_csv, index=False)
print("✅ Guardado:", out_poda_csv)

print("kept (head 5):")
display(pd.DataFrame({"kept": kept}).head(5))

print("dropped_log (head 5):")
pd.DataFrame(dropped_log).head(5)

Pool candidato: 229
Kept tras poda: 185
✅ Guardado: results_n02\poda_features_corr_log.csv
kept (head 5):


,kept
0,tasa_dolares_familias
1,operativa_total_mercado_de_cambios
2,derivados_financieros
3,exp_quimicos
4,dolar_mexico


dropped_log (head 5):


,dropped,kept,corr_abs,dropped_score,kept_score
0,uy_bcu_lrm_monto_propuesto_180d,uy_bcu_lrm_monto_aceptado_180d,0.923332,0.111148,0.167078
1,rem_gob,pas_gob,0.924889,0.108322,0.134171
2,inv_ANCAP,var_stock_pet,0.998419,0.056276,0.057048
3,cud_3anos,cud_2anos,0.951669,0.041505,0.047399
4,OFERTA_VENTA,OFERTA_COMPRA,0.946030,0.040915,0.041999


## 9) Information Value (IV) univariado sobre `kept`

Para cada horizonte `h`:
- target binario: `up_fwd_h`
- IV univariado por cuantiles
- guardamos una hoja por horizonte y una hoja agregada.

Mostramos las primeras 5 filas de `iv_agg`.

In [10]:
def iv_univariate(x: pd.Series, y: pd.Series, n_bins: int = 10) -> float:
    """IV clásico por bines (cuantiles). Univariado. y debe ser 0/1."""
    df_ = pd.DataFrame({"x": x, "y": y}).dropna()
    if df_.shape[0] < 200 or df_["y"].nunique() < 2:
        return np.nan

    try:
        df_["bin"] = pd.qcut(df_["x"], q=n_bins, duplicates="drop")
    except Exception:
        return np.nan

    grp = df_.groupby("bin", observed=True)["y"]
    good = (grp.count() - grp.sum()).astype(float)  # y=0
    bad  = (grp.sum()).astype(float)                # y=1

    # suavizado para evitar cero
    eps = 0.5
    good = good + eps
    bad  = bad + eps

    dist_good = good / good.sum()
    dist_bad  = bad / bad.sum()

    woe = np.log(dist_bad / dist_good)
    iv = ((dist_bad - dist_good) * woe).sum()
    return float(iv)

iv_tables = {}
iv_sum = pd.Series(0.0, index=kept)

for h in HORIZON_MAP.keys():
    yb = df[f"up_fwd_{h}"]
    rows = []
    for v in kept:
        rows.append((v, iv_univariate(df[v], yb, n_bins=10)))
    t = pd.DataFrame(rows, columns=["var", "iv"]).sort_values("iv", ascending=False).reset_index(drop=True)
    iv_tables[h] = t
    iv_sum = iv_sum.add(t.set_index("var")["iv"].fillna(0.0), fill_value=0.0)

iv_agg = pd.DataFrame({"var": iv_sum.index, "iv_mean": iv_sum.values / len(HORIZON_MAP)})
iv_agg = iv_agg.sort_values("iv_mean", ascending=False).reset_index(drop=True)

out_iv_xlsx = os.path.join(RESULTS_DIR, "information_value.xlsx")
with pd.ExcelWriter(out_iv_xlsx) as writer:
    for h, t in iv_tables.items():
        t.to_excel(writer, sheet_name=f"iv_{h}", index=False)
    iv_agg.to_excel(writer, sheet_name="iv_agregado", index=False)

print("✅ Guardado:", out_iv_xlsx)
iv_agg.head(5)

✅ Guardado: results_n02\information_value.xlsx


,var,iv_mean
0,pas_gob,1.699530
1,inversion_de_cartera,1.632307
2,tasa_dolares_familias,1.369506
3,IRAE,1.218480
4,exp_carne,1.197753


## 10) Random Forest simple sobre `kept` + importancias

- Split temporal simple: 70% train, 30% test
- RandomForestRegressor simple (sin tuning)
- Importancia:
  - SHAP si está disponible (sobre una submuestra del test)
  - si no, permutation importance

Guardamos `rf_importance_by_horizon.xlsx` y mostramos las primeras 5 filas.

In [11]:
import numpy as np
import pandas as pd
import os

from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer

# =========================
# RANDOM FOREST IMPORTANCE (simple, sin SHAP, sin leakage por imputación)
# - Split temporal 70/30 (df debe estar ordenado por fecha)
# - Imputación: fit SOLO en train, aplica en test
# - Importancia: permutation_importance sobre test (NO usar para "tunar" el modelo final)
# =========================

rf_importances = {}

# Asegurar que kept exista y sea lista
kept = list(kept)

for h, H in HORIZON_MAP.items():
    print(f"\n--- RF horizonte {h} ({H} días) ---")
    target_col = f"ret_fwd_{h}"

    # Quedarse solo con filas donde target existe + features (evita errores)
    cols_needed = kept + [target_col]
    df_rf = df[cols_needed].copy()
    df_rf = df_rf.dropna(subset=[target_col])

    # Split temporal 70/30
    split_idx = int(len(df_rf) * 0.7)
    train = df_rf.iloc[:split_idx]
    test  = df_rf.iloc[split_idx:]

    X_train = train[kept]
    y_train = train[target_col].astype(float)
    X_test  = test[kept]
    y_test  = test[target_col].astype(float)

    # Imputación SIN leakage: ajustar solo en train
    imputer = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=kept, index=X_train.index)
    X_test_imp  = pd.DataFrame(imputer.transform(X_test),  columns=kept, index=X_test.index)

    # Random Forest (parámetros conservadores y estables)
    rf = RandomForestRegressor(
        n_estimators=200,        # más estable que 80 para ranking
        max_depth=6,
        min_samples_leaf=20,
        max_features="sqrt",     # ayuda con colinealidad
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_imp, y_train)

    # Permutation importance en test (OOS)
    perm = permutation_importance(
        rf, X_test_imp, y_test,
        n_repeats=10,
        random_state=42,
        n_jobs=-1
    )

    imp = pd.Series(perm.importances_mean, index=kept).sort_values(ascending=False)
    rf_importances[h] = imp

# Unificar en DataFrame y guardar
rf_imp_df = pd.concat(rf_importances, axis=1)

out_rf = os.path.join(RESULTS_DIR, "rf_importance_by_horizon.xlsx")
rf_imp_df.to_excel(out_rf)
print("✅ Guardado:", out_rf)

rf_imp_df.head(10)


--- RF horizonte 1m (21 días) ---

--- RF horizonte 3m (63 días) ---

--- RF horizonte 6m (126 días) ---

--- RF horizonte 12m (252 días) ---
✅ Guardado: results_n02\rf_importance_by_horizon.xlsx


,1m,3m,6m,12m
EPU_br,0.053454,0.060171,0.024178,-0.004671
riesgo_sob,0.009310,0.003081,0.002446,-0.000436
exp_veh_y_trans,0.008383,-0.002871,-0.000244,-0.000067
imp_arg,0.005251,-0.002706,0.006969,0.002125
operativa_total_mercado_de_cambios,0.005032,0.002329,0.009760,0.009057
dolar_mexico,0.004491,0.037177,0.063778,0.044764
uy_bcu_lrm_monto_propuesto_360d,0.004372,0.005070,0.004206,-0.005601
exp_arroz,0.003546,-0.001867,-0.002191,0.004657
exp_quimicos,0.003290,0.013712,0.013296,0.017607
uy_bcu_lrm_monto_aceptado_360d,0.003282,0.001439,0.000540,-0.002525


## 11) Agregar importancias RF por bloque (corto vs largo)

- Corto: 1m + 3m
- Largo: 6m + 12m

Mostramos las primeras 5 filas de las series agregadas.

In [12]:
SHORT_H = ["1m", "3m"]
LONG_H  = ["6m", "12m"]

rf_short = rf_imp_df[SHORT_H].mean(axis=1)
rf_long  = rf_imp_df[LONG_H].mean(axis=1)

pd.DataFrame({"rf_short": rf_short, "rf_long": rf_long}).head(5)

,rf_short,rf_long
EPU_br,0.056813,0.009753
riesgo_sob,0.006195,0.001005
exp_veh_y_trans,0.002756,-0.000155
imp_arg,0.001272,0.004547
operativa_total_mercado_de_cambios,0.003680,0.009408


## 12) Score por bloque (corto vs largo)

Construimos scores por bloque usando:
- correlación forward (promedio dentro del bloque)
- IV (promedio dentro del bloque)
- RF importancia (promedio dentro del bloque)

y normalización por ranking a [0,1].

Mostramos las primeras 5 filas de `final_rank_short` y `final_rank_long`.

In [13]:
def rank_norm(s: pd.Series) -> pd.Series:
    r = s.rank(method="average")
    return (r - r.min()) / (r.max() - r.min() + 1e-9)

# Correlación por bloque (promedio de abs_corr)
corr_short = agg[agg["h"].isin(SHORT_H)].groupby("var")["abs_corr"].mean()
corr_long  = agg[agg["h"].isin(LONG_H)].groupby("var")["abs_corr"].mean()

# IV por bloque (usa iv_tables con hoja por horizonte)
iv_short = pd.concat([iv_tables[h].set_index("var")["iv"] for h in SHORT_H], axis=1).mean(axis=1)
iv_long  = pd.concat([iv_tables[h].set_index("var")["iv"] for h in LONG_H], axis=1).mean(axis=1)

# Alinear a kept
corr_short = corr_short.reindex(kept).fillna(0.0)
corr_long  = corr_long.reindex(kept).fillna(0.0)
iv_short   = iv_short.reindex(kept).fillna(0.0)
iv_long    = iv_long.reindex(kept).fillna(0.0)
rf_short_a = rf_short.reindex(kept).fillna(0.0)
rf_long_a  = rf_long.reindex(kept).fillna(0.0)

score_short = W_CORR*rank_norm(corr_short) + W_IV*rank_norm(iv_short) + W_IMP*rank_norm(rf_short_a)
score_long  = W_CORR*rank_norm(corr_long)  + W_IV*rank_norm(iv_long)  + W_IMP*rank_norm(rf_long_a)

final_rank_short = pd.DataFrame({
    "var": kept,
    "corr_short": corr_short.values,
    "iv_short": iv_short.values,
    "rf_short": rf_short_a.values,
    "score_short": score_short.values
}).sort_values("score_short", ascending=False).reset_index(drop=True)

final_rank_long = pd.DataFrame({
    "var": kept,
    "corr_long": corr_long.values,
    "iv_long": iv_long.values,
    "rf_long": rf_long_a.values,
    "score_long": score_long.values
}).sort_values("score_long", ascending=False).reset_index(drop=True)

display(final_rank_short.head(5))
display(final_rank_long.head(5))

,var,corr_short,iv_short,rf_short,score_short
0,EPU_br,0.252094,0.755116,0.056813,0.993478
1,dolar_mexico,0.250962,0.527133,0.020834,0.986957
2,exp_carne,0.188496,0.446872,0.013461,0.964674
3,operativa_total_mercado_de_cambios,0.269907,0.254826,0.003680,0.959783
4,tasa_dolares_familias,0.274437,0.507291,0.000711,0.958152


,var,corr_long,iv_long,rf_long,score_long
0,derivados_financieros,0.416052,1.019232,0.034828,0.984783
1,exp_carne,0.312629,1.948634,0.029141,0.976087
2,exp_arg,0.325190,1.322799,0.024470,0.976087
3,dolar_mexico,0.295082,1.093924,0.054271,0.969565
4,exp_quimicos,0.383849,0.979901,0.015451,0.968478


## 13) Jaccard entre selecciones corto vs largo

Seleccionamos Top-N por score y calculamos Jaccard.
Mostramos además las primeras 5 variables de cada lista.

In [14]:
vars_short = final_rank_short.head(TOP_N_BLOCK)["var"].tolist()
vars_long  = final_rank_long.head(TOP_N_BLOCK)["var"].tolist()

jaccard_sl = len(set(vars_short) & set(vars_long)) / len(set(vars_short) | set(vars_long))
print(f"Aproximadamente {jaccard_sl:.3f} del conjunto combinado de variables es compartido entre corto y largo.")

pd.DataFrame({"vars_short": vars_short[:5], "vars_long": vars_long[:5]})

Aproximadamente 0.471 del conjunto combinado de variables es compartido entre corto y largo.


,vars_short,vars_long
0,EPU_br,derivados_financieros
1,dolar_mexico,exp_carne
2,exp_carne,exp_arg
3,operativa_total_mercado_de_cambios,dolar_mexico
4,tasa_dolares_familias,exp_quimicos


## 14) Export de rankings por bloque

Guardamos un Excel con:
- ranking corto
- ranking largo

Mostramos las primeras 5 filas del ranking corto.

In [15]:
out_rank_blocks = os.path.join(RESULTS_DIR, "ranking_corto_largo.xlsx")
with pd.ExcelWriter(out_rank_blocks) as writer:
    final_rank_short.to_excel(writer, sheet_name="ranking_corto", index=False)
    final_rank_long.to_excel(writer, sheet_name="ranking_largo", index=False)

print("✅ Guardado:", out_rank_blocks)
final_rank_short.head(5)

✅ Guardado: results_n02\ranking_corto_largo.xlsx


,var,corr_short,iv_short,rf_short,score_short
0,EPU_br,0.252094,0.755116,0.056813,0.993478
1,dolar_mexico,0.250962,0.527133,0.020834,0.986957
2,exp_carne,0.188496,0.446872,0.013461,0.964674
3,operativa_total_mercado_de_cambios,0.269907,0.254826,0.003680,0.959783
4,tasa_dolares_familias,0.274437,0.507291,0.000711,0.958152


## 15) Crear datasets finales (corto y largo) y exportar

Construimos:
- `df_short`: fecha + variables corto
- `df_long`: fecha + variables largo

Exportamos a Excel y mostramos las primeras 5 filas de cada uno.

In [16]:
df_short = df[[DATE_COL] + vars_short].copy()
df_long  = df[[DATE_COL] + vars_long].copy()

In [17]:
# =========================================================
# AJUSTE – Asegurar que el target base (usd_dlog) quede presente
# en los datasets finales para que el pipeline posterior pueda
# reconstruir targets forward si lo necesita.
# =========================================================

TARGET_VAR = "usd_dlog"
assert TARGET_VAR in df.columns, "usd_dlog no está en df"

for _df_name, _df in [("df_short", df_short), ("df_long", df_long)]:
    if TARGET_VAR not in _df.columns:
        _df[TARGET_VAR] = df[TARGET_VAR]

print("✅ usd_dlog asegurado en df_short y df_long.")


✅ usd_dlog asegurado en df_short y df_long.


In [18]:
# Export datasets corto y largo (XLSX + opcional CSV)

def ensure_target_first(df_in: pd.DataFrame, date_col: str, target_col: str) -> pd.DataFrame:
    cols = list(df_in.columns)
    if target_col in cols and date_col in cols:
        ordered = [date_col, target_col] + [c for c in cols if c not in [date_col, target_col]]
        return df_in[ordered].copy()
    return df_in.copy()

out_short_xlsx = os.path.join(RESULTS_DIR, "df_st_daily_short_plazo.xlsx")
out_long_xlsx  = os.path.join(RESULTS_DIR, "df_st_daily_long_plazo.xlsx")

df_short = ensure_target_first(df_short, DATE_COL, "usd_dlog")
df_long  = ensure_target_first(df_long,  DATE_COL, "usd_dlog")

df_short.to_excel(out_short_xlsx, index=False)
df_long.to_excel(out_long_xlsx, index=False)

if EXPORT_CSV:
    out_short_csv = os.path.join(RESULTS_DIR, "df_st_daily_short_plazo.csv")
    out_long_csv  = os.path.join(RESULTS_DIR, "df_st_daily_long_plazo.csv")
    df_short.to_csv(out_short_csv, index=False)
    df_long.to_csv(out_long_csv, index=False)

print("✅ Guardados datasets corto y largo:")
print(" -", out_short_xlsx)
print(" -", out_long_xlsx)
if EXPORT_CSV:
    print(" -", out_short_csv)
    print(" -", out_long_csv)


✅ Guardados datasets corto y largo:
 - results_n02\df_st_daily_short_plazo.xlsx
 - results_n02\df_st_daily_long_plazo.xlsx
 - results_n02\df_st_daily_short_plazo.csv
 - results_n02\df_st_daily_long_plazo.csv


## 15bis) Crear dataset general y exportar

Construimos un dataset único (general) para probar un solo set de variables en todos los horizontes.

Por defecto:
- tomamos el ranking global por correlación promedio (`abs_corr_mean`) dentro de `kept` (ya podado por redundancia)
- seleccionamos `TOP_N_GENERAL`

Exportamos a Excel y (opcionalmente) CSV.

In [19]:
# Ranking global (promedio de abs_corr) restringido a `kept`
rank_global = (agg.groupby("var")["abs_corr"].mean()
               .reindex(kept)
               .sort_values(ascending=False)
               .reset_index())
rank_global.columns = ["var", "abs_corr_mean"]

vars_general = rank_global.head(TOP_N_GENERAL)["var"].tolist()

df_general = df[[DATE_COL] + vars_general].copy()

# Forzar inclusión del target base
TARGET_VAR = "usd_dlog"
if TARGET_VAR not in df_general.columns:
    df_general[TARGET_VAR] = df[TARGET_VAR]

# Reordenar columnas: fecha, target, features
cols_general = [DATE_COL, TARGET_VAR] + [c for c in df_general.columns if c not in [DATE_COL, TARGET_VAR]]
df_general = df_general[cols_general].copy()

out_general_xlsx = os.path.join(RESULTS_DIR, "df_st_daily_general.xlsx")
df_general.to_excel(out_general_xlsx, index=False)

if EXPORT_CSV:
    out_general_csv = os.path.join(RESULTS_DIR, "df_st_daily_general.csv")
    df_general.to_csv(out_general_csv, index=False)

print("✅ Guardado dataset general:")
print(" -", out_general_xlsx)
if EXPORT_CSV:
    print(" -", out_general_csv)

df_general.head(5)


✅ Guardado dataset general:
 - results_n02\df_st_daily_general.xlsx
 - results_n02\df_st_daily_general.csv


,Fecha,usd_dlog,tasa_dolares_familias,operativa_total_mercado_de_cambios,derivados_financieros,exp_quimicos,dolar_mexico,exp_carne,EPU_br,uy_bcu_lrm_pct_adjudicado_180d,...,cud_1ano,cud_6meses,TPM_br,ust_6meses,cud_2anos,ust_1ano,tasa_pesos_familias,uy_bcu_lrm_tasa_corte_360d,FedFunds,OFERTA_COMPRA
0,2015-04-01,-0.000778,7.3,633684.12,13.08,39065,15.01,168829,317.27,0.57,...,0.04,0.06,0.0,-0.01,0.04,-0.01,39.36,0.00,0.01,-0.03
1,2015-04-06,-0.002727,7.3,633684.12,13.08,39065,14.92,168829,317.27,0.57,...,0.04,0.04,0.0,0.01,0.01,-0.01,39.36,0.00,0.00,-0.08
2,2015-04-07,0.005059,7.3,633684.12,13.08,39065,14.91,168829,317.27,0.57,...,-0.03,-0.04,0.0,-0.02,0.00,-0.02,39.36,0.00,0.00,0.14
3,2015-04-08,0.008889,7.3,633684.12,13.08,39065,15.08,168829,317.27,0.57,...,-0.04,-0.05,0.0,0.02,-0.07,0.03,39.36,0.00,0.00,0.23
4,2015-04-09,0.006902,7.3,633684.12,13.08,39065,15.22,168829,317.27,0.57,...,-0.04,-0.04,0.0,0.00,-0.01,0.01,39.36,0.07,0.00,0.18


**Comparación de variables**

In [20]:
# ===============================
# 16) Comparación de variables: corto vs largo
# ===============================

set_short = set(vars_short)
set_long  = set(vars_long)

vars_common     = sorted(list(set_short & set_long))
vars_only_short = sorted(list(set_short - set_long))
vars_only_long  = sorted(list(set_long - set_short))

print(f"Variables corto plazo: {len(set_short)}")
print(f"Variables largo plazo: {len(set_long)}")
print(f"Variables comunes: {len(vars_common)}")
print(f"Solo corto plazo: {len(vars_only_short)}")
print(f"Solo largo plazo: {len(vars_only_long)}")

df_vars_compare = pd.DataFrame({
    "var": vars_common + vars_only_short + vars_only_long,
    "en_corto_plazo": (
        [1]*len(vars_common) +
        [1]*len(vars_only_short) +
        [0]*len(vars_only_long)
    ),
    "en_largo_plazo": (
        [1]*len(vars_common) +
        [0]*len(vars_only_short) +
        [1]*len(vars_only_long)
    ),
    "categoria": (
        ["comun"]*len(vars_common) +
        ["solo_corto"]*len(vars_only_short) +
        ["solo_largo"]*len(vars_only_long)
    )
})

# Ordenar para que quede prolijo
df_vars_compare = df_vars_compare.sort_values(
    ["categoria", "var"]
).reset_index(drop=True)

# Export
out_vars = os.path.join(RESULTS_DIR, "variables_corto_vs_largo.xlsx")
df_vars_compare.to_excel(out_vars, index=False)

print("✅ Guardado:", out_vars)

df_vars_compare.head(10)

Variables corto plazo: 50
Variables largo plazo: 50
Variables comunes: 32
Solo corto plazo: 18
Solo largo plazo: 18
✅ Guardado: results_n02\variables_corto_vs_largo.xlsx


,var,en_corto_plazo,en_largo_plazo,categoria
0,CANTIDAD,1,1,comun
1,CANTIDAD_OPERACIONES,1,1,comun
2,EPU_br,1,1,comun
3,IMESI,1,1,comun
4,IRAE,1,1,comun
5,VIX,1,1,comun
6,cud_10anos,1,1,comun
7,cud_6meses,1,1,comun
8,derivados_financieros,1,1,comun
9,dolar_euro,1,1,comun


**Exportar variables de cada dataset**

In [21]:
# ==========================================
# EXPORTAR VARIABLES EN UN SOLO EXCEL
# (Hojas separadas y orden alfabético)
# ==========================================

import pandas as pd

# Ordenar alfabéticamente
short_sorted = sorted(vars_short)
long_sorted = sorted(vars_long)
general_sorted = sorted(vars_general)

# Crear dataframes
df_short = pd.DataFrame({"variable": short_sorted})
df_long = pd.DataFrame({"variable": long_sorted})
df_general = pd.DataFrame({"variable": general_sorted})

# Exportar a un solo archivo Excel
with pd.ExcelWriter("variables_por_dataset.xlsx") as writer:
    df_short.to_excel(writer, sheet_name="Short", index=False)
    df_long.to_excel(writer, sheet_name="Long", index=False)
    df_general.to_excel(writer, sheet_name="General", index=False)

print("Archivo 'variables_por_dataset.xlsx' exportado correctamente.")

Archivo 'variables_por_dataset.xlsx' exportado correctamente.
